In [ ]:
def asset_location(
    allocation_dict,
    tickers_by_tax_efficiency,
    accounts_by_tax_advantage,
    total_money,
    account_limits,
    current_holdings=None,
    lock_taxable=True
):
    """
    Optimize asset location by placing least tax-efficient assets in most tax-advantaged accounts.
    
    Parameters:
    -----------
    allocation_dict : dict
        Dictionary of {ticker: percentage} where percentage is decimal (e.g., 0.30 for 30%)
    tickers_by_tax_efficiency : list
        List of tickers ordered from most to least tax efficient
    accounts_by_tax_advantage : list
        List of account names ordered from most to least tax advantaged
    total_money : float
        Total amount of money to allocate
    account_limits : dict
        Dictionary of {account_name: limit}. Use float('inf') for no limit
    current_holdings : dict, optional
        Current holdings as nested dict: {account: {ticker: amount}}
        Holdings in taxable accounts are locked (no selling to avoid capital gains)
        Holdings in Roth accounts can be rebalanced freely (tax-free)
    lock_taxable : bool, default True
        If True, locks taxable account holdings (no selling to avoid capital gains)
        If False, allows rebalancing in taxable accounts (may trigger capital gains)
    
    Returns:
    --------
    dict
        Nested dictionary: {account: {ticker: amount}}
    """
    import pandas as pd
    
    # Initialize tracking
    locked_holdings = {}  # Holdings we can't sell (taxable accounts)
    
    # If current_holdings is empty or all amounts are zero, treat as fully liquid
    all_zero = True
    if current_holdings:
        for account, holdings in current_holdings.items():
            if holdings and any(amount > 0 for amount in holdings.values()):
                all_zero = False
                break
    if not current_holdings or all_zero:
        # Treat as fully liquid, ignore lock_taxable and holdings
        current_holdings = {account: {} for account in accounts_by_tax_advantage}
        lock_taxable = False
    
    # Process current holdings if provided
    if current_holdings:
        print(f"\n{'='*70}")
        print(f"Processing existing holdings...")
        print(f"{'='*70}")
        
        for account, holdings in current_holdings.items():
            if not holdings:
                continue
                
            # Check if this is a taxable account (not Roth)
            is_taxable = 'roth' not in account.lower()
            
            if is_taxable and lock_taxable:
                # Lock in taxable holdings - we can't sell these
                locked_holdings[account] = holdings.copy()
                total_locked = sum(holdings.values())
                print(f"  🔒 Locked: {account} - ${total_locked:,.2f} (avoiding capital gains)")
                for ticker, amount in holdings.items():
                    print(f"      {ticker}: ${amount:,.2f}")
            else:
                total_flexible = sum(holdings.values())
                status = "🔄 Flexible" if not is_taxable else "⚠️  Unlocked"
                note = "(can rebalance tax-free)" if not is_taxable else "(taxable - may trigger capital gains)"
                print(f"  {status}: {account} - ${total_flexible:,.2f} {note}")
                for ticker, amount in holdings.items():
                    print(f"      {ticker}: ${amount:,.2f}")
    
    # Calculate dollar amounts for each ticker based on total
    ticker_amounts = {ticker: total_money * pct for ticker, pct in allocation_dict.items()}
    
    # Initialize result with locked holdings and adjust remaining capacity
    result = {account: {} for account in accounts_by_tax_advantage}
    remaining_capacity = account_limits.copy()
    
    # Pre-fill locked holdings and reduce capacity
    for account, holdings in locked_holdings.items():
        result[account] = holdings.copy()
        total_locked = sum(holdings.values())
        remaining_capacity[account] -= total_locked
    
    # Adjust ticker amounts: subtract what we already have locked in taxable
    ticker_amounts_to_allocate = ticker_amounts.copy()
    for account, holdings in locked_holdings.items():
        for ticker, amount in holdings.items():
            if ticker in ticker_amounts_to_allocate:
                ticker_amounts_to_allocate[ticker] -= amount
                ticker_amounts_to_allocate[ticker] = max(0, ticker_amounts_to_allocate[ticker])
    
    print(f"\n{'='*70}")
    print("Allocating remaining amounts...")
    print(f"{'='*70}")
    
    # Reverse the tax efficiency list to go from least to most tax efficient
    tickers_least_to_most_efficient = list(reversed(tickers_by_tax_efficiency))
    
    # Allocate each ticker starting with least tax efficient
    for ticker in tickers_least_to_most_efficient:
        if ticker not in ticker_amounts_to_allocate:
            continue
            
        amount_to_place = ticker_amounts_to_allocate[ticker]
        
        if amount_to_place <= 0:
            continue
        
        # Try to place in accounts from most to least tax advantaged
        for account in accounts_by_tax_advantage:
            if amount_to_place <= 0:
                break
                
            available_space = remaining_capacity[account]
            amount_placed = min(amount_to_place, available_space)
            
            if amount_placed > 0:
                if ticker in result[account]:
                    result[account][ticker] += amount_placed
                else:
                    result[account][ticker] = amount_placed
                    
                remaining_capacity[account] -= amount_placed
                amount_to_place -= amount_placed
    
    return result


In [8]:
# ==============================================================================
# CONFIGURATION
# ==============================================================================

# Target allocation (percentages must sum to 1.0)
allocation_dict = {
    'SPMO': 0.2, 
    'VOO': 0.195,
    'SSO': 0.195, 
    'AVNM': 0.1,
    'AVUV': 0.1, 
    'AVDV': 0.07,
    'AVEM': 0.07,
    'IDMO': 0.07
}

# Order tickers by tax efficiency (most efficient first)
tickers_by_tax_efficiency = [
    'VOO',    # Most tax efficient (low turnover, low dividends)
    'AVNM',
    'AVEM',
    'AVDV',
    'AVUV',
    'IDMO',
    'SPMO',
    'SSO',   # Least tax efficient (higher turnover, more dividends, more growth)
]

# Order accounts by tax advantage (most advantaged first)
accounts_by_tax_advantage = [
    'Roth IRA',
    'Taxable Brokerage'
]

# Current holdings (modify this dict to test different scenarios)
# Empty dict means starting fresh
current_holdings = {
    'Roth IRA': {},
    'Taxable Brokerage': {}
}
manual_roth = True
if manual_roth:
    roth_sum = 15128
else:
    roth_sum = sum(current_holdings['Roth IRA'].values())

# Account limits (derived from current holdings)
account_limits = {
    'Roth IRA': roth_sum,
    'Taxable Brokerage': float('inf')
}

# Total portfolio value (calculated from current holdings)
total_money = sum(sum(holdings.values()) for holdings in current_holdings.values())

# ==============================================================================
# RUN ALLOCATION
# ==============================================================================

result = asset_location(
    allocation_dict,
    tickers_by_tax_efficiency,
    accounts_by_tax_advantage,
    total_money,
    account_limits,
    current_holdings=current_holdings,
    lock_taxable=False
)

print_asset_location(result, total_money, title="Target Allocation")



Processing existing holdings...

Allocating remaining amounts...


KeyError: 'Amount'

In [ ]:
# ==============================================================================
# CALCULATE REBALANCING (if you have existing holdings)
# ==============================================================================

# If you already have positions and want to know what to buy/sell:
trades, summary = calculate_rebalance(result, current_holdings, total_money, rebalance_threshold=0.02)
print_rebalance(trades, summary, total_money)



Rebalancing Actions Required



,Account,Ticker,Action,Amount,Percentage
1,Taxable Brokerage,SCHG,BUY,"$3,839.00",10.00%
0,Taxable Brokerage,VOO,SELL,"$3,756.50",9.79%



----------------------------------------------------------------------
Total to BUY:  $3,839.00
Total to SELL: $3,756.50
Net Change:    $82.50
----------------------------------------------------------------------


In [4]:
# ==============================================================================
# SAVE/LOAD STATE (Optional - for persistence)
# ==============================================================================

# Save current allocation to CSV
# save_portfolio_state(result, csv_filepath='asset_allocation_state.csv')

# Load existing allocation from CSV
# loaded_holdings = load_portfolio_state(csv_filepath='asset_allocation_state.csv')
